In [96]:
!pip install azureml-sdk

In [97]:
import os
import json
import requests

from azureml.core import Workspace
from azureml.core.model import Model
from azureml.core.environment import Environment
from azureml.core.conda_dependencies import CondaDependencies
from azureml.core.model import InferenceConfig
from azureml.core.webservice import AciWebservice, Webservice

In [98]:
config_file_path = "/content/azure_config.json"

with open(config_file_path, 'r') as file:
    data = json.load(file)

subscription_id = data["subscription_id"]
resource_group = data["resource_group"]
workspace_name = data["workspace_name"]
region = data["region"]

In [99]:
try:
    ws = Workspace(subscription_id=subscription_id, resource_group=resource_group, workspace_name=workspace_name)
    print(f'Workspace {workspace_name} found.')
except Exception as e:
    ws = Workspace.create(name=workspace_name,
                          subscription_id=subscription_id,
                          resource_group=resource_group,
                          location=region)
    print(f'Workspace {workspace_name} created.')


Workspace image-super-resolution-ws found.


In [100]:
model_path = '/content/models/RRDB_ESRGAN_x4.pth'
model_name='image-super-resolution'

In [101]:
registered_models = Model.list(workspace=ws)
model_already_registered = any(model.name == model_name for model in registered_models)

if model_already_registered:
    print(f"Model '{model_name}' already exists in the workspace.")
else:
    registered_model = Model.register(model_path=model_path, model_name=model_name, workspace=ws)
    print(f"Model '{model_name}' registered successfully.")

Model 'image-super-resolution' already exists in the workspace.


In [102]:
conda_env = Environment('my-conda-env')

conda_packages = [
    'python=3.8',
    'pytorch',
    'torchvision',
    'numpy',
    'pillow'
]

conda_deps = CondaDependencies.create(conda_packages=conda_packages)

conda_env.python.conda_dependencies = conda_deps

In [103]:
inference_config = InferenceConfig(source_directory='src', entry_script='score.py', environment=conda_env)

In [104]:
aci_config = AciWebservice.deploy_configuration(cpu_cores=1, memory_gb=1)

In [105]:
existing_services = Webservice.list(workspace=ws)
service_already_exists = any(service.name == 'image-super-resolution' for service in existing_services)

if service_already_exists:
    existing_service = Webservice(workspace=ws, name='image-super-resolution')
    existing_service.delete()
    print("Existing service 'image-super-resolution' deleted.")

service = Model.deploy(workspace=ws,
                       name='image-super-resolution',
                       models=[registered_model],
                       inference_config=inference_config,
                       deployment_config=aci_config)
service.wait_for_deployment(show_output=True)
print("Service 'image-super-resolution' deployed successfully.")

Running
2024-04-03 12:01:26+00:00 Check and wait for operation (2862baf0-70d2-483d-818e-1f3060010e62) to finish.
2024-04-03 12:01:30+00:00 Deleting service entity.
Succeeded
Existing service 'image-super-resolution' deleted.


<ipython-input-105-cf78e9792603>:9: FutureWarning: azureml.core.model:
To leverage new model deployment capabilities, AzureML recommends using CLI/SDK v2 to deploy models as online endpoint, 
please refer to respective documentations 
https://docs.microsoft.com/azure/machine-learning/how-to-deploy-managed-online-endpoints /
https://docs.microsoft.com/azure/machine-learning/how-to-attach-kubernetes-anywhere 
For more information on migration, see https://aka.ms/acimoemigration 
To disable CLI/SDK v1 deprecation warning set AZUREML_LOG_DEPRECATION_WARNING_ENABLED to 'False'
  service = Model.deploy(workspace=ws,


Tips: You can try get_logs(): https://aka.ms/debugimage#dockerlog or local deployment: https://aka.ms/debugimage#debug-locally to debug if deployment takes longer than 10 minutes.
Running
2024-04-03 12:01:45+00:00 Creating Container Registry if not exists.
2024-04-03 12:01:47+00:00 Use the existing image.
2024-04-03 12:01:47+00:00 Generating deployment configuration.
2024-04-03 12:01:47+00:00 Submitting deployment to compute.
2024-04-03 12:01:55+00:00 Checking the status of deployment image-super-resolution..
2024-04-03 12:02:58+00:00 Checking the status of inference endpoint image-super-resolution.
Succeeded
ACI service creation operation finished, operation "Succeeded"
Service 'image-super-resolution' deployed successfully.


In [106]:
# print(service.get_logs())

In [107]:
scoring_uri = service.scoring_uri
scoring_uri

'http://b004fdf5-a3e3-4bc3-b75e-d1d929a1d3d2.centralindia.azurecontainer.io/score'